In [ ]:
# --- Setup: make the `ecp` support package available -----------------
# Colab opens a single notebook and installs nothing, so fetch `ecp` from
# the public repo if it isn't importable yet. On Binder/local it is already
# installed, so this cell is a fast no-op there.
try:
    import ecp  # noqa: F401
except ModuleNotFoundError:
    import subprocess, sys
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q",
         "git+https://github.com/ramador09/elementary-computational-physics-binder@main"],
        check=True,
    )


# 7.23 Second Quantization: The Occupation Number Becomes the State

<!-- This single H1 (one per notebook, "# <number> <Title>") is the page's
     title: it sets the sidebar entry, breadcrumb, browser tab, and search
     result. The branded banner below is generated by the shared `ecp`
     package, so the look of every notebook in the series lives in one place. -->

In [ ]:
from ecp.style import header, use_style

use_style()  # apply the series Matplotlib style
header(
    volume="Volume VII — Quantum Statistical Mechanics",
    number="7.23",
    title="Second Quantization: The Occupation Number Becomes the State",
    blurb="This volume computed with occupation lists from the start; what it never had "
    "were operators that change the list. Here they are: ladders that add and "
    "remove quanta, strings that carry antisymmetry so nobody has to, and a Fock "
    "space where indistinguishability is automatic instead of enforced. We verify "
    "the same physics twice — labels-then-erase against never-label, to sixteen "
    "digits — rebuild the volume's founding factorization in one line, and then "
    "solve the smallest interesting interacting problem there is: two sites, two "
    "electrons, and the origin of magnetism hiding in a six-by-six matrix.",
    difficulty="advanced",
    estimate="200–240 min",
    optional=True,
)

## Notebook overview

The Coda opens here: three optional notebooks — "The Many-Body Gateway" — that sit after
the volume's arc (which [§7.22](eigenstate-thermalization.ipynb) closed) and outside it, for readers who want the formalism
the volume kept gesturing toward. Nothing downstream depends on them; everything in them
depends on what the volume already built.

This first notebook gives the volume's deepest bookkeeping habit its own operators. From
[§7.7](bose-einstein-fermi-dirac.ipynb) onward, a quantum gas was never "particle 1 here, particle 2 there": it was a *list of
occupations*, one integer per mode, and every free partition function factorized because
the list did. What the volume never had was a way to *change* the list — operators that
add and remove quanta, so that tunneling, interaction, and pairing can be written without
ever naming a particle. First quantization names the particles and then labors to erase
the names: the $N!$ of [§7.8](classical-limit-thermal-wavelength.ipynb), the sign gymnastics of every Slater determinant, labels
introduced only to be crossed out. Second quantization never introduces them, and
indistinguishability stops being a constraint to enforce and becomes a fact of the
notation.

The construction is deliberately computational. Bosons come first because Volume VI
already built their machinery (the harmonic-oscillator ladder matrices, reused verbatim),
and they arrive with the artifact every truncation carries: on a truncated ladder the
canonical commutator is exact everywhere *except the top state*, and the corner is
demonstrated before anything is trusted. Fermions expose the real subtlety: the naive
multi-mode construction *fails*, demonstrably — operators on different sites commute, and
commuting creators build symmetric states, bosons wearing fermion costumes. The repair is
the Jordan–Wigner string, presented not as the solving trick of [§7.19](transverse-field-ising.ipynb) but as the *definition* of
lattice fermions, with the full anticommutator table verified at machine precision as the
unit test no sign bug survives. The epistemic center is a **rendezvous**: the same
two-fermion problem built twice, first-quantized (antisymmetrized pair basis,
Slater–Condon signs by hand) and second-quantized (a number block of Fock space), spectra
agreeing to sixteen digits — a theorem checked to the bit, and a sign audit in which the
by-hand signs are exactly the labor the strings automate. The volume's cornerstone is then
re-derived in one line (the factorization of [§7.7](bose-einstein-fermi-dirac.ipynb) as a Fock-space grand trace), and the notation
proves its worth on the **Hubbard dimer** — the smallest interesting interacting problem
and the hydrogen molecule in disguise: exact ground-state formula matched to all digits,
Hellmann–Feynman confirmed as a numerical theorem, superexchange measured against
$4t^2/U$, quantum chemistry's founding debate resolved by a $6\times6$ matrix, and a
two-peak specific heat showing one interaction manufacturing two thermodynamic scales.
A Bogoliubov–de Gennes gesture closes the loop of [§7.19](transverse-field-ising.ipynb): the crown of Movement V re-read as a
change of basis on quadratic fermions.

> **Conventions (this notebook).** Modes are ordered left to right in every Kronecker
> chain: mode 0 is the leftmost (most significant) factor, and the single-mode fermion
> matrices act on the per-mode basis $(|0\rangle, |1\rangle)$ with
> $c = \begin{psmallmatrix}0&1\\0&0\end{psmallmatrix}$ (so $c|1\rangle = |0\rangle$).
> The Jordan–Wigner string puts $Z$ factors on all modes *left* of the target:
> $c_i = Z^{\otimes i} \otimes c \otimes I^{\otimes(M-i-1)}$ — stated once here, and
> audited by the anticommutator gate after every construction. Truncated boson ladders
> are `numpy.diag(numpy.sqrt(numpy.arange(1, nmax+1)), 1)` with the top-state corner
> checked, always. Number sectors are extracted with `numpy.ix_` on the diagonal of
> $\hat N$ (project the Hamiltonian, never diagonalize the full Fock matrix for one
> sector). The Hubbard dimer's mode order is $[1\!\uparrow, 1\!\downarrow, 2\!\uparrow,
> 2\!\downarrow]$. Every spectrum is computed once and reused (the cache discipline of [§7.22](eigenstate-thermalization.ipynb)).
> Hellmann–Feynman derivatives use a central difference with step $10^{-4}$, both
> evaluations in the *same* number sector. The BdG block matrix is
> `numpy.block([[A, B], [-B, -A]])` with the antiperiodic wrap-link sign.
>
> **How to read the checks.** Each exercise closes with a `validate` call against an
> independent fact: the commutator table against $\delta_{ij}$ (and the corner against
> $-n_{\max}$); the anticommutator table against the fermion algebra; sector dimensions
> against binomial coefficients; two constructions of one problem against each other; the
> Fock grand trace against the product of [§7.7](bose-einstein-fermi-dirac.ipynb); the dimer against its exact formula, its
> derivative theorem, and its two scales; the BdG spectrum against Pfeuty. A ✓ is strong
> evidence; a ✗ is a prompt to *locate the discrepancy*.
>
> **Scope.** The Hubbard model at scale (chains, lattices, the metal–insulator problem:
> Essler et al., *The One-Dimensional Hubbard Model*), superconductivity's mean-field
> theory and the Kitaev chain (BdG at scale) are outward horizons; Green's functions and
> linear response are the Coda's next two notebooks. See Fetter & Walecka, *Quantum
> Theory of Many-Particle Systems* (Ch. 1–2, the formalism at full depth); Ashcroft &
> Mermin Ch. 32 (exchange); Heitler & London 1927. Cross-reference [§7.7](bose-einstein-fermi-dirac.ipynb) (the occupation
> habit and the factorization, re-derived below), [§7.8](classical-limit-thermal-wavelength.ipynb) (the $N!$, retired), [§7.19](transverse-field-ising.ipynb)/[§7.20](imaginary-time-quantum-classical.ipynb)
> (Jordan–Wigner and parity, recast as principle), [§7.22](eigenstate-thermalization.ipynb) (the sector discipline), Volume
> VI (the ladder matrices, reused), and forward to [§7.24](greens-functions.ipynb) (the propagator at temperature;
> the dimer's Green's function promised) and [§7.25](linear-response-kubo.ipynb) (response).

## Theory in brief

### Two ways to not know who's who

Indistinguishability admits two bookkeeping schemes, and the volume has so far only used
the expensive one. First quantization writes $N$-particle states as functions of labeled
coordinates and then (anti)symmetrizes: $N!$ terms per state (the factor of [§7.8](classical-limit-thermal-wavelength.ipynb), there
repaired by hand), a sign per permutation, and the permanent burden of never letting a
label acquire meaning. Second quantization starts from the observation the volume has
been making since [§7.7](bose-einstein-fermi-dirac.ipynb) — for identical particles, the physical state is fully specified by
*how many* quanta occupy each mode — and takes it literally:

```{math}
:label: eq-two-quantizations
|\Psi\rangle \;=\; |n_0, n_1, n_2, \dots\rangle,
\qquad
n_k \in \{0, 1, 2, \dots\} \ \text{(bosons)},
\quad
n_k \in \{0, 1\} \ \text{(fermions)} .
```

The occupation list *is* the state; the space spanned by all lists is **Fock space**; and
no particle is ever named, so there is nothing to erase. What this notation needs — and
what this notebook builds as explicit matrices — are operators that change the list.

### Bosons: ladders, kron, and the truncation corner

Volume VI already built the single-mode operators: the harmonic oscillator's ladder
matrices, with $a^\dagger|n\rangle = \sqrt{n+1}\,|n{+}1\rangle$. A computer must truncate
the infinite ladder at some $n_{\max}$, and the truncation leaves a fingerprint worth
demonstrating before anything else, because every truncated-boson computation inherits
it:

```{math}
:label: eq-boson-ladders
[a, a^\dagger] = \mathbb{1}
\quad\text{exactly}\quad
\textit{except}\ \langle n_{\max}|[a, a^\dagger]|n_{\max}\rangle = -n_{\max},
```

since the top state has nowhere to go: $a^\dagger|n_{\max}\rangle$ is silently the zero
vector, and the commutator's diagonal reads $(1, 1, \dots, 1, -n_{\max})$. The standing
rule issued below: audit the top state's occupancy in every truncated-boson computation.
Multi-mode operators are Kronecker products with identities, and for bosons that naive
recipe is the correct one — different modes commute, as they should.

### Fermions: the sign catastrophe, and the string that is a definition

For fermions the naive recipe fails, and the failure is the lesson. The single-mode
matrix $c = \begin{psmallmatrix}0&1\\0&0\end{psmallmatrix}$ is trivial, but embedding it
as $I \otimes \cdots \otimes c \otimes \cdots \otimes I$ makes operators on *different*
modes commute — and commuting creation operators build symmetric states: bosons in
fermion costume (demonstrated below: $\|\{c_0, c_1^\dagger\}\| = 2$ where the algebra
demands zero). The repair is the Jordan–Wigner string, and the honest framing is that it
is not a repair at all but the **definition** of what fermionic operators are on a
tensor-product space ([§7.19](transverse-field-ising.ipynb) used it as a solving trick; here it graduates to principle):

```{math}
:label: eq-jw-definition
c_i \;=\; \underbrace{Z \otimes \cdots \otimes Z}_{i\ \text{factors}}
\,\otimes\, c \,\otimes\, I \otimes \cdots \otimes I,
\qquad
\{c_i, c_j^\dagger\} = \delta_{ij},
\quad
\{c_i, c_j\} = 0 .
```

The string of $Z$'s counts the parity of occupied modes to the left, which is exactly the
$(-1)^{\text{position}}$ that first quantization carries in its Slater signs. The
anticommutator table on the right is the **unit test of all fermionic code**: no string
ordering or sign convention bug survives it, so it is run after every operator
construction in this notebook (Fetter & Walecka Ch. 1 develop the formalism from the
algebra up).

### Number blocks

Every Hamiltonian in this notebook conserves particle number, and the consequence is
the resolve-the-symmetry discipline of [§7.19](transverse-field-ising.ipynb)/[§7.22](eigenstate-thermalization.ipynb) arriving in its native habitat. With
$\hat N = \sum_i c_i^\dagger c_i$ diagonal in the occupation basis,

```{math}
:label: eq-number-blocks
[H, \hat N] = 0
\quad\Longrightarrow\quad
H = \bigoplus_{n=0}^{M} H^{(n)},
\qquad
\dim H^{(n)} = \binom{M}{n},
```

the Fock matrix block-diagonalizes by particle number, with sector dimensions that are
binomial coefficients (verified below by bit census). The practical rule: extract the
wanted block with `numpy.ix_` on $\hat N$'s diagonal and diagonalize *that* — projecting
the Hamiltonian is quadratic bookkeeping, while diagonalizing the full Fock matrix for
one sector wastes a factor that grows exponentially.

### The rendezvous: one theorem, checked to the bit

That the two bookkeeping schemes describe the same physics is a theorem every text states
and almost none checks numerically; this notebook's epistemic center is checking it to
machine precision. The same problem — two spinless fermions on four sites, hopping $t$
and nearest-neighbor repulsion $V$ — is built twice:

```{math}
:label: eq-sq-rendezvous
\underbrace{\text{6 antisymmetric pair states, Slater–Condon signs by hand}}_{\text{first-quantized}}
\quad\overset{!}{=}\quad
\underbrace{\text{the } N = 2 \text{ block of the 16-state Fock space}}_{\text{second-quantized}} .
```

The first-quantized route needs the one-difference sign rule (the alignment sign that
kills most hand computations, stated explicitly below); the second-quantized route needs
no signs at all, because the strings carry them. Agreement to sixteen digits is therefore
also a **sign audit**: the by-hand Slater signs are exactly the labor Jordan–Wigner
automates. A boson counterpart (two bosons, three sites, on-site $U$) is specified and
assigned to the gate — expected on the same theorem, confirmed there rather than here.

### [§7.7](bose-einstein-fermi-dirac.ipynb), re-derived in one line

The volume's founding factorization was stated in [§7.7](bose-einstein-fermi-dirac.ipynb) for independently filling modes and
has powered every quantum gas since. In Fock space it is one line: for a free Hamiltonian
the grand trace factorizes mode by mode,

```{math}
:label: eq-factorization-rederived
\Xi \;=\; \sum_{\text{all Fock states}} e^{-\beta(E - \mu N)}
\;=\; \prod_k \big(1 + z\,e^{-\beta\varepsilon_k}\big)
\qquad (\text{fermions};\ z = e^{\beta\mu}),
```

because the exponential of a sum over modes is a product over modes and each fermionic
mode contributes its two occupations. The left side is computed below by brute force
(all 16 states of a four-mode problem) and meets the right side at fourteen digits: [§7.7](bose-einstein-fermi-dirac.ipynb)
was second quantization all along, with the operators kept offstage.

### The Hubbard dimer (centerpiece)

Interactions are precisely what breaks the product above, and the smallest interesting
case is two sites and two electrons — which is also, read chemically, the hydrogen
molecule in a minimal basis. The model every many-body course meets first:

```{math}
:label: eq-hubbard-dimer
H \;=\; -t\sum_{\sigma}\big(c^\dagger_{1\sigma}c_{2\sigma} + \text{h.c.}\big)
\;+\; U\big(n_{1\uparrow}n_{1\downarrow} + n_{2\uparrow}n_{2\downarrow}\big),
\qquad
E_0 = \frac{U - \sqrt{U^2 + 16t^2}}{2},
```

with the exact ground-state formula from diagonalizing the singlet $3\times3$ block (the
derivation is two lines once the $N = 2$ sector is in hand; Ashcroft & Mermin Ch. 32
give the physics of the exchange it contains). The program below: exact ED against the
formula at $U/t = 0, 4, 8, 20$; **Hellmann–Feynman as a numerical theorem**
($dE_0/dU = \langle\sum_i n_{i\uparrow}n_{i\downarrow}\rangle$, stated and then checked,
the course's way); the crossover read through double occupancy and
$\langle\mathbf S_1\!\cdot\!\mathbf S_2\rangle$; the $U = 20t$ level diagram with the
triplet *exactly* at zero; **superexchange** measured against $4t^2/U$ (virtual double
occupation buys the singlet its energy: magnetism from kinetic exchange); the
molecular-orbital versus Heitler–London reading (1927's founding debate, resolved by a
$6\times6$ matrix); and a Volume-VII-native result — the canonical $C(T)$ of the sector
showing **two peaks**, the spin scale $J = 4t^2/U$ and the charge scale $U$, two
thermodynamic scales manufactured by one interaction.

### The BdG gesture

The Jordan–Wigner image of [§7.19](transverse-field-ising.ipynb) contained more than hopping: it contained *pairing* terms
$c^\dagger c^\dagger$ that create and destroy fermions in pairs. Quadratic Hamiltonians
of that kind are diagonalized by a canonical (Bogoliubov) transformation, organized as a
block matrix on doubled space:

```{math}
:label: eq-bdg
H = \sum_{ij}\Big[c_i^\dagger A_{ij} c_j
+ \tfrac12\big(c_i^\dagger B_{ij} c_j^\dagger + \text{h.c.}\big)\Big]
\;\longrightarrow\;
\mathcal{H}_{\mathrm{BdG}} = \begin{pmatrix} A & B \\ -B & -A \end{pmatrix},
```

with $A$ symmetric, $B$ antisymmetric, and eigenvalues in $\pm\varepsilon$ pairs. Built
in real space for the chain of [§7.19](transverse-field-ising.ipynb), with the antiperiodic wrap-link sign (the parity
bookkeeping of [§7.19](transverse-field-ising.ipynb)/[§7.20](imaginary-time-quantum-classical.ipynb), third appearance), its spectrum lands on Pfeuty's
$\pm\varepsilon(k)$ at machine precision: the crown of Movement V, re-read as a change
of basis in the space this notebook built. "Exactly solvable" acquires its precise
meaning — quadratic in some fermions.

### The gateway reading

The operators built here are an alphabet. The Coda's next notebook teaches the sentences
— the Green's function, which adds one quantum to a many-body system and honestly
reports where it goes ([§7.24](greens-functions.ipynb)) — and the one after asks equilibrium a question and gets an
answer (the linear response of [§7.25](linear-response-kubo.ipynb)).

## Setup

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from itertools import combinations
from scipy.signal import argrelmax

from ecp import draw, validate

ACCENT, INK, SOFT = draw.ACCENT, draw.INK, draw.SOFT
RED = "#c1121f"

# Conventions: mode 0 is the LEFTMOST kron factor; per-mode basis (|0>, |1>)
# with c|1> = |0>; the JW string puts Z on every mode left of the target.
# Stated once here; the anticommutator gate audits it after every build.
C1 = np.array([[0.0, 1.0], [0.0, 0.0]])  # single-mode annihilator
Z1 = np.array([[1.0, 0.0], [0.0, -1.0]])
I1 = np.eye(2)


def kron_chain(ops):
    """Kronecker product of a list of matrices, left to right (mode 0 leftmost)."""
    out = np.array([[1.0]])
    for o in ops:
        out = np.kron(out, o)
    return out


def boson_ladder(nmax):
    """Truncated single-mode boson annihilator on the (nmax+1)-state ladder.

    a = numpy.diag(numpy.sqrt(numpy.arange(1, nmax+1)), 1): a|n> = sqrt(n)|n-1>.
    The truncation leaves the canonical commutator exact everywhere EXCEPT the
    top state, where [a, a+] reads -nmax (Exercise 1 demonstrates): every
    truncated-boson computation inherits this corner, so the standing rule is
    to audit the top state's occupancy, always.

    Parameters
    ----------
    nmax : int
        Highest occupation kept.

    Returns
    -------
    numpy.ndarray
        The (nmax+1) x (nmax+1) annihilator.
    """
    return np.diag(np.sqrt(np.arange(1.0, nmax + 1)), 1)


def naive_fermions(M):
    """The WRONG multi-mode fermions: bare kron embeddings, no strings.

    c_i = I x ... x c x ... x I. Operators on different modes COMMUTE, so
    their creators build symmetric states: bosons in fermion costume.
    Exercise 2 demonstrates the failure before repairing it; kept as a
    helper because the failure is the lesson.

    Parameters
    ----------
    M : int
        Number of modes.

    Returns
    -------
    list of numpy.ndarray
        The naive annihilators.
    """
    return [kron_chain([I1] * i + [C1] + [I1] * (M - i - 1)) for i in range(M)]


def jw_ops(M):
    """Jordan-Wigner fermion annihilators: c_i = Z x...x Z x c x I x...x I.

    The string of Z's on the modes LEFT of the target counts the parity of
    occupied modes before position i — exactly the sign first quantization
    carries in its Slater determinants. This is the DEFINITION of lattice
    fermions on a tensor-product space (§7.19 used it as a solving trick;
    here it is the construction itself).

    Parameters
    ----------
    M : int
        Number of modes.

    Returns
    -------
    list of numpy.ndarray
        The M annihilators, mode order as in the convention block.
    """
    return [kron_chain([Z1] * i + [C1] + [I1] * (M - i - 1)) for i in range(M)]


def anticommutator_gate(cs):
    """The unit test of fermionic code: max deviation of the full algebra table.

    Checks {c_i, c_j+} = delta_ij and {c_i, c_j} = 0 for every pair. A wrong
    string order, a flipped kron convention, or a stray sign CANNOT hide from
    this table, which is why it runs after every operator construction in
    the notebook.

    Parameters
    ----------
    cs : list of numpy.ndarray
        Candidate annihilators.

    Returns
    -------
    float
        Maximum absolute deviation from the fermion algebra.
    """
    D = cs[0].shape[0]
    dev = 0.0
    for i, ci in enumerate(cs):
        for j, cj in enumerate(cs):
            dev = max(dev, np.abs(ci @ cj.T + cj.T @ ci - np.eye(D) * (i == j)).max())
            dev = max(dev, np.abs(ci @ cj + cj @ ci).max())
    return float(dev)


def number_operator(cs):
    """N = sum_i c_i+ c_i — diagonal in the occupation basis."""
    return sum(c.T @ c for c in cs)


def sector_block(H, n_diag, n):
    """Extract the n-particle block of a Fock-space matrix via numpy.ix_.

    Project the HAMILTONIAN, never the eigenvectors: pulling one binomial
    block costs quadratic bookkeeping, while diagonalizing the full Fock
    matrix to use one sector wastes an exponentially growing factor.

    Parameters
    ----------
    H : numpy.ndarray
        Fock-space operator.
    n_diag : numpy.ndarray
        Integer diagonal of the number operator (rounded, never cast).
    n : int
        Particle number wanted.

    Returns
    -------
    idx : numpy.ndarray
        The sector's Fock indices.
    block : numpy.ndarray
        The projected matrix.
    """
    idx = np.where(n_diag == n)[0]
    return idx, H[np.ix_(idx, idx)]


def canonical_C(evals, Ts):
    """Canonical heat capacity C(T) = Var(E)/T^2 from a sector spectrum.

    Ground-shifted weights e^{-beta(E - E_0)} (the discipline of §7.4) so no
    underflow at any temperature on the grid.

    Parameters
    ----------
    evals : numpy.ndarray
        Sector eigenvalues.
    Ts : numpy.ndarray
        Temperature grid.

    Returns
    -------
    numpy.ndarray
        C(T) on the grid.
    """
    C = np.empty_like(Ts)
    for k, T in enumerate(Ts):
        w = np.exp(-(evals - evals[0]) / T)
        Zc = w.sum()
        Em = np.sum(w * evals) / Zc
        E2 = np.sum(w * evals**2) / Zc
        C[k] = (E2 - Em**2) / T**2
    return C


def bdg_matrix(A, B):
    """The Bogoliubov-de Gennes block matrix numpy.block([[A, B], [-B, -A]]).

    For H = sum c+ A c + (1/2)(c+ B c+ + h.c.) with A symmetric and B
    antisymmetric, the eigenvalues come in +-epsilon pairs: the canonical
    transformation that diagonalizes quadratic fermions, organized as one
    matrix on doubled space.

    Parameters
    ----------
    A : numpy.ndarray
        Symmetric hopping/field block.
    B : numpy.ndarray
        Antisymmetric pairing block.

    Returns
    -------
    numpy.ndarray
        The 2M x 2M BdG matrix.
    """
    return np.block([[A, B], [-B, -A]])


# The four-mode fermion workspace used throughout (16 Fock states), built
# once and reused — the cache discipline of §7.22 in one line.
M_MODES = 4
CS = jw_ops(M_MODES)
N_OP = number_operator(CS)
N_DIAG = np.round(np.diag(N_OP)).astype(int)  # round, NEVER cast (Exercise 1)
print(
    f"workspace: {M_MODES} JW modes, {1 << M_MODES} Fock states, "
    f"anticommutator gate = {anticommutator_gate(CS):.1e}"
)

## Exercise 1 — Ladders without labels

Bosons first: the machinery Volume VI already built, with the artifact every truncation
carries. Cite {eq}`eq-two-quantizations`, {eq}`eq-boson-ladders`.

1. State the two bookkeeping schemes (labels-then-erase versus never-label) and the cost
   of the first, with the $N!$ of [§7.8](classical-limit-thermal-wavelength.ipynb) recalled in one line.
2. Build `boson_ladder(nmax)` and two-mode operators by `numpy.kron`; verify
   $[a_i, a_j^\dagger] = \delta_{ij}$ away from the corner.
3. Demonstrate the truncation corner (the commutator's diagonal $(1, \dots, 1,
   -n_{\max})$) and issue the rule: audit the top state's occupancy in every
   truncated-boson computation.
4. Record the meta-trap in one dry line: an integer cast once turned
   $0.9999999999999996$ into $0$ in this notebook's own verification — round, never
   cast.

In [ ]:
# (solution hidden on the public site)


### Validation 1

In [ ]:
validate.check(
    off_corner_dev < 1e-14
    and abs(corner_diag[-1] + NMAX) < 1e-12
    and cross_dev < 1e-14,
    "the ladder and its edge: canonical off the corner, -nmax on it, modes independent",
    f"corner = {corner_diag[-1]:.0f} at nmax = {NMAX}",
)

## Exercise 2 — The sign catastrophe, and the string that is a definition

Naive fermion krons build bosons in costume; Jordan–Wigner is not a trick. Cite
{eq}`eq-jw-definition`.

1. Build the naive multi-mode fermions with `naive_fermions` and demonstrate the failure:
   $\|\{c_0, c_1^\dagger\}\|$ where the algebra demands zero, and why commuting creators
   build symmetric states.
2. Define the Jordan–Wigner operators (`jw_ops`, convention stated once in Setup) and run
   `anticommutator_gate`: the full table $\{c_i, c_j^\dagger\} = \delta_{ij}$,
   $\{c_i, c_j\} = 0$ at machine precision.
3. Recast [§7.19](transverse-field-ising.ipynb) (prose): the string is the *definition* of lattice fermions on a
   tensor-product space — the antisymmetry first quantization carries in signs, carried
   here by operators.
4. Issue the standing rule (prose): the anticommutator table is the unit test of all
   fermionic code; no ordering or sign bug survives it — run it after every construction.

In [ ]:
# (solution hidden on the public site)


In [ ]:
# (solution hidden on the public site)


### Validation 2

In [ ]:
validate.check(
    abs(naive_cross - 2.0) < 1e-12 and jw_table_dev < 1e-13,
    "fermions defined, not tricked: the naive failure at 2.0, the JW table at machine zero",
    f"naive ||{{c0, c1+}}|| = {naive_cross:.1f}, JW table {jw_table_dev:.1e}",
)

## Exercise 3 — Number blocks

The symmetry discipline of [§7.19](transverse-field-ising.ipynb)/[§7.22](eigenstate-thermalization.ipynb), in its native habitat. Cite {eq}`eq-number-blocks`.

1. Build $\hat N$ with `number_operator` and verify $[H, \hat N] = 0$ for this notebook's
   Hamiltonians (the rendezvous chain of Exercise 4 and the Hubbard dimer of Exercise 6).
2. Verify the sector dimensions against binomial coefficients by bit census, and
   visualize the block sparsity of a Fock Hamiltonian with states ordered by particle
   number.
3. Use `sector_block` (`numpy.ix_`) and state the practical rule: project the
   Hamiltonian, never diagonalize the full Fock matrix for one sector.
4. Connect in one line each: the parity sectors of [§7.19](transverse-field-ising.ipynb) and the resolve-every-symmetry
   rule of [§7.22](eigenstate-thermalization.ipynb) — the same bookkeeping, now the formalism's backbone.

In [ ]:
# (solution hidden on the public site)


In [ ]:
# (solution hidden on the public site)


### Validation 3

In [ ]:
validate.check(
    sector_dims == binomials,
    "blocks the size of binomials",
    f"{sector_dims} (bit census) = C(4, n)",
)
validate.check(
    comm_chain < 1e-13 and comm_dimer < 1e-13,
    "and both Hamiltonians commute with N at machine precision",
    f"chain {comm_chain:.1e}, dimer {comm_dimer:.1e}",
)

## Exercise 4 — The rendezvous: one theorem, sixteen digits (centerpiece)

The same two fermions, built with labels and without. Cite {eq}`eq-sq-rendezvous`.

1. Build the problem first-quantized: two spinless fermions on four sites ($t = 1$,
   nearest-neighbor $V = 1.7$), the $\binom{4}{2} = 6$ antisymmetric pair states
   $|ij\rangle$ ($i < j$), matrix elements by the Slater–Condon one-difference rule with
   the alignment sign stated explicitly.
2. Build it second-quantized: the Fock Hamiltonian of Exercise 3, projected to $N = 2$ by
   `sector_block`.
3. Verify the six eigenvalues agree at machine precision and read it correctly (prose): a
   theorem checked to the bit, and a sign audit — the by-hand Slater signs are exactly
   what the strings automate.
4. Boson counterpart, assigned to the gate (expected, not pre-verified): two bosons on
   three sites with on-site $U$ — symmetric-projector pair basis against the
   truncated-kron Fock build at $n_{\max} = 2$ (the corner rule applied), the gate
   confirming equality.

In [ ]:
# (solution hidden on the public site)


In [ ]:
# (solution hidden on the public site)


### Validation 4

In [ ]:
validate.close(
    E_FQ,
    E_SQ,
    "labels-then-erase = never-label, to the bit",
    atol=1e-12,
)
validate.close(
    E_B_FOCK,
    E_B_FQ,
    "and the assigned boson counterpart confirms the theorem in the symmetric channel",
    atol=1e-12,
)

## Exercise 5 — [§7.7](bose-einstein-fermi-dirac.ipynb), re-derived in one line

The volume's founding factorization, recovered from the new language. Cite
{eq}`eq-factorization-rederived`.

1. Take the four free fermion modes of the rendezvous chain ($V = 0$: single-particle
   energies from `numpy.linalg.eigvalsh` of the hopping matrix) and compute the exact
   grand trace $\Xi = \sum e^{-\beta(E - \mu N)}$ over all 16 Fock states, sector by
   sector.
2. Verify against the product of [§7.7](bose-einstein-fermi-dirac.ipynb) $\prod_k (1 + z\,e^{-\beta\varepsilon_k})$ at
   $\beta = 1.3$, $\mu = 0.4$.
3. Say what happened (prose): [§7.7](bose-einstein-fermi-dirac.ipynb) was second quantization with the operators offstage —
   the factorization is the statement that free modes fill independently, and Fock space
   is where that sentence is grammatical.
4. One breath outward: interactions are precisely what breaks the product — the next
   exercise builds the smallest one.

In [ ]:
# (solution hidden on the public site)


### Validation 5

In [ ]:
validate.close(
    Xi_fock,
    Xi_prod,
    "the cornerstone, from the new language",
    rtol=1e-12,
)

## Exercise 6 — The Hubbard dimer (centerpiece)

Two sites, two electrons, the origin of magnetism — and H$_2$ in disguise. Cite
{eq}`eq-hubbard-dimer`.

1. Build `hubbard_dimer(t, U)` (mode order $[1\!\uparrow, 1\!\downarrow, 2\!\uparrow,
   2\!\downarrow]$), extract the $N = 2$ sector with `sector_block`, and verify ED
   against the exact $E_0 = (U - \sqrt{U^2 + 16t^2})/2$ at $U/t = 0, 4, 8, 20$.
2. Verify Hellmann–Feynman as a numerical theorem: $dE_0/dU$ by central difference (step
   $10^{-4}$, both evaluations in the same sector — the trap stated) against
   $\langle\sum_i n_{i\uparrow}n_{i\downarrow}\rangle$ from the ground eigenvector; plot
   $\langle d\rangle(U)$ from the molecular-orbital value downward and
   $\langle\mathbf S_1\!\cdot\!\mathbf S_2\rangle(U)$ from $-3/8$ toward the singlet's
   $-3/4$.
3. Read the $U = 20t$ level diagram (singlet, triplet exactly at zero — explain why in
   one sentence — doublons near $U$) and measure superexchange $E_T - E_S$ against
   $4t^2/U$, deriving the exact splitting and its second-order reading.
4. Tell 1927 in two sentences (molecular orbital versus Heitler–London, the dimer as the
   interpolation) and deliver the Volume-VII-native result: the canonical $C(T)$ of the
   sector at $U = 8t$ (`canonical_C`, peaks located by `scipy.signal.argrelmax`) with two
   peaks — the spin scale and the charge scale.

In [ ]:
# (solution hidden on the public site)


In [ ]:
# (solution hidden on the public site)


In [ ]:
# (solution hidden on the public site)


### Validation 6

In [ ]:
validate.check(
    e0_dev < 1e-12,
    "the dimer meets its exact formula at every U",
    f"max |ED - exact| = {e0_dev:.1e} over U/t = 0, 4, 8, 20",
)
validate.check(
    hf_max_dev < 1e-6,
    "Hellmann-Feynman confirmed as a numerical theorem",
    f"max |dE0/dU - <d>| = {hf_max_dev:.1e} (central difference, step 1e-4, same sector)",
)
validate.check(
    abs(E20[1]) < 1e-12 and abs(E20[2]) < 1e-12 and abs(E20[3]) < 1e-12,
    "the triplet sits exactly at zero (double occupation Pauli-forbidden)",
    f"triplet energies {np.round(E20[1:4], 14)}",
)
validate.close(
    J_meas,
    J_exact,
    "superexchange measured against the exact splitting",
    rtol=1e-10,
)
validate.close(
    J_meas,
    J_pert,
    "and its leading 4t^2/U reading holds to one percent at U = 20t",
    rtol=2e-2,
)
validate.close(
    ss_20,
    -0.7427,
    "the bond approaches the Heitler-London singlet",
    rtol=1e-3,
)
validate.check(
    len(C_peaks) == 2 and C_peaks[1] / C_peaks[0] > 15.0,
    "two thermodynamic scales from one interaction",
    f"C(T) peaks at {C_peaks[0]:.3f} (spin, J = {J_scale}) and {C_peaks[1]:.2f} (charge, U = 8)",
)

## Exercise 7 — (STUDENT/STRETCH) The BdG gesture: [§7.19](transverse-field-ising.ipynb) in its native language

Pairing terms, one block matrix, and the crown re-read. Cite {eq}`eq-bdg`.

1. Write the TFIM's Jordan–Wigner image (hopping plus pairing $c^\dagger c^\dagger$
   terms) as the quadratic form of {eq}`eq-bdg` and organize it with `bdg_matrix`
   ($A$ symmetric, $B$ antisymmetric — stated and checked).
2. Build it in real space for $M = 8$, $J = 1$, $h = 1.3$ with the antiperiodic
   wrap-link sign (the parity bookkeeping of [§7.19](transverse-field-ising.ipynb)/[§7.20](imaginary-time-quantum-classical.ipynb), third appearance).
3. Verify: the eigenvalues are the $\pm\varepsilon(k)$ pairs of Pfeuty's dispersion at
   the antiperiodic momenta.
4. Close the loop (prose): Movement V's magic was a canonical transformation on
   quadratic fermions — a change of basis in the space this notebook built; "exactly
   solvable" acquires its precise meaning, and the integrability-as-memory of [§7.22](eigenstate-thermalization.ipynb) gains its
   algebraic face in one line.

In [ ]:
# (solution hidden on the public site)


In [ ]:
# (solution hidden on the public site)


### Validation 7

In [ ]:
validate.close(
    ev_bdg,
    pm_pfeuty,
    "the crown as a change of basis",
    atol=1e-12,
)
validate.check(
    sym_check < 1e-14 and antisym_check < 1e-14,
    "with the quadratic form's symmetries as stated (A symmetric, B antisymmetric)",
    f"A: {sym_check:.1e}, B: {antisym_check:.1e}",
)

## Exercise 8 — (Synthesis) The alphabet

No new computation: what changed today, and what did not.

This notebook changed no physics and changed everything about how physics gets written.
The volume's occupation lists gained the operators that move them; antisymmetry moved out
of hand-managed signs and into strings that cannot forget; a theorem the reader may never
have doubted was nonetheless checked to sixteen digits, because that is the house style;
and the founding factorization of Movement II turned out to be one line long in the right
notation. Then the notation proved its worth on the smallest problem it improves: two
electrons on two sites, where the competition between motion and repulsion produces — out
of nothing but kinetic energy denied — an antiferromagnetic bond, a separation of
thermodynamic scales, and both sides of quantum chemistry's founding argument as limits
of one six-by-six matrix.

Notation is never neutral. First quantization makes the easy things easy and the true
things exhausting; second quantization makes indistinguishability — the truest thing in
this volume — free. One measures a formalism by what it renders unthinkable to get wrong:
after today, forgetting a minus sign is not a mistake one can make, because there is no
minus sign to forget.

The operators built here are an alphabet. Next the Coda teaches the sentences: the
Green's function, which asks what happens when one quantum is added to a many-body system
and honestly reports where it goes ([§7.24](greens-functions.ipynb)).

## Notebook summary

The Coda's opening notebook: the volume's occupation habit, given operators.

- **Two bookkeeping schemes** {eq}`eq-two-quantizations`: labels-then-erase (the $N!$ of [§7.8](classical-limit-thermal-wavelength.ipynb),
  Slater signs) versus never-label (Fock states as occupation lists); the second is
  built, the first is retired.
- **Bosons** {eq}`eq-boson-ladders`: Volume VI's ladder matrices, kron-composed; the
  truncation corner demonstrated ($[a, a^\dagger]$ diagonal $(1, \dots, 1, -n_{\max})$,
  gated) and the standing rule issued (audit the top state, always); the
  round-never-cast meta-trap recorded in one dry line.
- **Fermions** {eq}`eq-jw-definition`: the naive kron failure demonstrated
  ($\|\{c_0, c_1^\dagger\}\| = 2$: bosons in costume, gated) and repaired by the
  Jordan–Wigner string as *definition*; the anticommutator table verified at machine
  zero (gated) and issued as the unit test of all fermionic code.
- **Number blocks** {eq}`eq-number-blocks`: $[H, \hat N] = 0$ (gated); sector dimensions
  $= \binom{M}{n}$ by bit census (gated); `numpy.ix_` projection and the
  project-then-diagonalize rule.
- **The rendezvous** {eq}`eq-sq-rendezvous`: two fermions built with Slater–Condon signs by
  hand and with strings; spectra equal to $10^{-15}$ (gated) — a theorem checked to the
  bit and a sign audit; the assigned boson counterpart confirmed in the symmetric
  channel (gated).
- **[§7.7](bose-einstein-fermi-dirac.ipynb) re-derived** {eq}`eq-factorization-rederived`: the 16-state Fock grand trace
  meets $\prod_k(1 + ze^{-\beta\varepsilon_k})$ at $10^{-14}$ (gated) — the cornerstone
  was second quantization with the operators offstage.
- **The Hubbard dimer** {eq}`eq-hubbard-dimer`: ED on the exact formula at four
  couplings (gated at $10^{-13}$); Hellmann–Feynman as a numerical theorem (gated at
  $10^{-6}$); the crossover in $\langle d\rangle$ and
  $\langle\mathbf S_1\!\cdot\!\mathbf S_2\rangle \to -0.7427$ (gated); the triplet
  exactly at zero (gated) with superexchange $0.19804$ against the exact splitting and
  its $4t^2/U$ reading (both gated); 1927 resolved as a crossover; the two-peak $C(T)$
  (spin scale $J$, charge scale $U$; gated).
- **BdG** {eq}`eq-bdg`: the $[[A, B], [-B, -A]]$ block with the antiperiodic wrap sign
  reproduces Pfeuty's $\pm\varepsilon(k)$ at machine precision (gated): [§7.19](transverse-field-ising.ipynb) as a
  canonical transformation, and integrability's algebraic face named.

Standing rules issued here: audit the truncation corner; round, never cast; run the
anticommutator table after every fermionic construction; project the Hamiltonian, not
the eigenvectors; keep both sides of a Hellmann–Feynman difference in the same sector.

## Outlook

- **Green's functions** ([§7.24](greens-functions.ipynb)): the propagator at temperature; the Lehmann
  representation computed exactly, Matsubara sums finally at work, and the dimer's
  Green's function as the worked interacting example.
- **Linear response** ([§7.25](linear-response-kubo.ipynb)): equilibrium answering questions — Kubo, checked against
  real-time dynamics.
- **The Hubbard model at scale**: chains, lattices, the metal–insulator problem (Essler
  et al., *The One-Dimensional Hubbard Model*; outward).
- **BdG at scale**: superconductivity's mean-field theory; the Kitaev chain (outward,
  named).
- Cross-reference: [§7.7](bose-einstein-fermi-dirac.ipynb)/[§7.8](classical-limit-thermal-wavelength.ipynb) (the habit, and the retired $N!$), [§7.19](transverse-field-ising.ipynb)/[§7.20](imaginary-time-quantum-classical.ipynb) (Jordan–Wigner
  and parity, recast as principle), [§7.22](eigenstate-thermalization.ipynb) (the sector discipline; integrability's
  algebraic face), Volume VI (the ladders, reused).

In [ ]:
from ecp.style import footer

footer()